[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/langchain-ai/langchain-academy/blob/main/module-1/chain.ipynb) [![Open in LangChain Academy](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66e9eba12c7b7688aa3dbb5e_LCA-badge-green.svg)](https://academy.langchain.com/courses/take/intro-to-langgraph/lessons/58238466-lesson-4-chain)

# Chain

## 回顾

我们构建了一个包含节点、普通边和条件边的简单图。

## 目标

现在，让我们构建一个简单的链，它结合了4个 [概念](https://python.langchain.com/v0.2/docs/concepts/)：

* 使用 [聊天消息](https://python.langchain.com/v0.2/docs/concepts/#messages) 作为我们的图状态
* 在图节点中使用 [聊天模型](https://python.langchain.com/v0.2/docs/concepts/#chat-models)
* 将 [绑定工具](https://python.langchain.com/v0.2/docs/concepts/#tools) 到我们的聊天模型
* 在图节点中 [执行工具调用](https://python.langchain.com/v0.2/docs/concepts/#functiontool-calling)

![Screenshot 2024-08-21 at 9.24.03 AM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab08dd607b08df5e1101_chain1.png)

In [ ]:
%%capture --no-stderr
%pip install --quiet -U langchain_openai langchain_core langgraph

## 消息

聊天模型可以使用 [`消息`](https://python.langchain.com/v0.2/docs/concepts/#messages)，这些消息捕获对话中的不同角色。

LangChain 支持各种消息类型，包括 `HumanMessage`、`AIMessage`、`SystemMessage` 和 `ToolMessage`。

这些分别代表来自用户的消息、来自聊天模型的消息、指导聊天模型行为的消息，以及来自工具调用的消息。

让我们创建一个消息列表。

每个消息可以提供以下几个内容：

* `content` - 消息内容
* `name` - 可选，消息作者
* `response_metadata` - 可选，元数据字典（例如，通常由模型提供商为 `AIMessages` 填充）

In [ ]:
from pprint import pprint
from langchain_core.messages import AIMessage, HumanMessage

messages = [AIMessage(content=f"所以你说你在研究海洋哺乳动物？", name="Model")]
messages.append(HumanMessage(content=f"是的，没错。",name="Lance"))
messages.append(AIMessage(content=f"太好了，你想了解什么。", name="Model"))
messages.append(HumanMessage(content=f"我想了解在美国观看虎鲸的最佳地点。", name="Lance"))

for m in messages:
    m.pretty_print()

## 聊天模型

[聊天模型](https://python.langchain.com/v0.2/docs/concepts/#chat-models) 可以使用消息序列作为输入，并支持消息类型，如上所述。

有 [很多](https://python.langchain.com/v0.2/docs/concepts/#chat-models) 可供选择！让我们使用 OpenAI。

让我们检查您的 `OPENAI_API_KEY` 是否已设置，如果没有，将要求您输入。

In [ ]:
import os, getpass

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"{var}: ")

_set_env("OPENAI_API_KEY")

我们可以加载一个聊天模型并用我们的消息列表调用它。

我们可以看到结果是一个带有特定 `response_metadata` 的 `AIMessage`。

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o")
result = llm.invoke(messages)
type(result)

In [ ]:
result

In [ ]:
result.response_metadata

## 工具

当您希望模型与外部系统交互时，工具非常有用。

外部系统（例如，API）通常需要特定的输入模式或有效载荷，而不是自然语言。

当我们绑定一个 API 作为工具时，我们给模型提供了所需输入模式的感知。

模型将根据用户的自然语言输入选择调用工具。

并且，它将返回一个符合工具模式的输出。

[许多 LLM 提供商支持工具调用](https://python.langchain.com/v0.1/docs/integrations/chat/)，LangChain 中的 [工具调用接口](https://blog.langchain.dev/improving-core-tool-interfaces-and-docs-in-langchain/) 很简单。

您可以简单地将任何 Python `函数` 传递到 `ChatModel.bind_tools(function)` 中。

![Screenshot 2024-08-19 at 7.46.28 PM.png](https://cdn.prod.website-files.com/65b8cd72835ceeacd4449a53/66dbab08dc1c17a7a57f9960_chain2.png)

让我们展示一个工具调用的简单示例！

`multiply` 函数是我们的工具。

In [ ]:
def multiply(a: int, b: int) -> int:
    """将 a 和 b 相乘。

    Args:
        a: 第一个整数
        b: 第二个整数
    """
    return a * b

llm_with_tools = llm.bind_tools([multiply])

如果我们传递一个输入 - 例如，`"2 乘以 3 是多少"` - 我们会看到返回一个工具调用。

工具调用具有与我们函数的输入模式匹配的特定参数，以及要调用的函数名称。

```
{'arguments': '{"a":2,"b":3}', 'name': 'multiply'}
```

In [ ]:
tool_call = llm_with_tools.invoke([HumanMessage(content=f"2 乘以 3 是多少", name="Lance")])

In [ ]:
tool_call.tool_calls

## 使用消息作为状态

有了这些基础，我们现在可以在图状态中使用 [`消息`](https://python.langchain.com/v0.2/docs/concepts/#messages)。

让我们将状态 `MessagesState` 定义为具有单个键的 `TypedDict`：`messages`。

`messages` 只是一个消息列表，如我们上面定义的（例如，`HumanMessage` 等）。

In [ ]:
from typing_extensions import TypedDict
from langchain_core.messages import AnyMessage

class MessagesState(TypedDict):
    messages: list[AnyMessage]

## 归约器

现在，我们有一个小问题！

如我们所讨论的，每个节点将为我们的状态键 `messages` 返回一个新值。

但是，这个新值会 [覆盖](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers) 之前的 `messages` 值。

随着图的运行，我们希望将消息 **追加** 到我们的 `messages` 状态键。

我们可以使用 [归约器函数](https://langchain-ai.github.io/langgraph/concepts/low_level/#reducers) 来解决这个问题。

归约器允许我们指定如何执行状态更新。

如果没有指定归约器函数，那么假定对键的更新应该 *覆盖它*，如我们之前看到的。

但是，要追加消息，我们可以使用预构建的 `add_messages` 归约器。

这确保任何消息都会追加到现有的消息列表中。

我们只需要用 `add_messages` 归约器函数作为元数据来注释我们的 `messages` 键。

In [ ]:
from typing import Annotated
from langgraph.graph.message import add_messages

class MessagesState(TypedDict):
    messages: Annotated[list[AnyMessage], add_messages]

由于在图状态中拥有消息列表是如此常见，LangGraph 有一个预构建的 [`MessagesState`](https://langchain-ai.github.io/langgraph/concepts/low_level/#messagesstate)！

`MessagesState` 的定义是：

* 具有预构建的单个 `messages` 键
* 这是一个 `AnyMessage` 对象列表
* 它使用 `add_messages` 归约器

我们通常使用 `MessagesState`，因为它比定义自定义 `TypedDict` 更简洁，如上所示。

In [ ]:
from langgraph.graph import MessagesState

class MessagesState(MessagesState):
    # 添加除 messages 之外需要的任何键，messages 是预构建的
    pass

为了更深入地了解，我们可以看到 `add_messages` 归约器如何独立工作。

In [ ]:
# 初始状态
initial_messages = [AIMessage(content="你好！我能如何帮助你？", name="Model"),
                    HumanMessage(content="我在寻找海洋生物学的信息。", name="Lance")
                   ]

# 要添加的新消息
new_message = AIMessage(content="当然，我可以帮你。你对什么具体感兴趣？", name="Model")

# 测试
add_messages(initial_messages , new_message)

## 我们的图

现在，让我们使用 `MessagesState` 与图。

In [ ]:
from IPython.display import Image, display
from langgraph.graph import StateGraph, START, END
    
# 节点
def tool_calling_llm(state: MessagesState):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

# 构建图
builder = StateGraph(MessagesState)
builder.add_node("tool_calling_llm", tool_calling_llm)
builder.add_edge(START, "tool_calling_llm")
builder.add_edge("tool_calling_llm", END)
graph = builder.compile()

# 查看
display(Image(graph.get_graph().draw_mermaid_png()))

如果我们传入 `你好！`，LLM 会在没有任何工具调用的情况下响应。

In [ ]:
messages = graph.invoke({"messages": HumanMessage(content="你好！")})
for m in messages['messages']:
    m.pretty_print()

当 LLM 确定输入或任务需要工具提供的功能时，它会选择使用工具。

In [ ]:
messages = graph.invoke({"messages": HumanMessage(content="将 2 和 3 相乘")})
for m in messages['messages']:
    m.pretty_print()